# ARCHS4 saturation: pathway recovery against model rank K

Coverage asks how much *data* CLAMP needs; this asks how much *model capacity*, and whether the two interact. Subsampling is by study, the same draws coverage uses, so K is the only thing varying within a fraction. Recovery uses FDR 0.05 throughout, recomputed here from each ORA's saved per-LV results.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
    library(data.table)
    library(ggplot2)
    library(ggrepel)
    library(here)
})

SATURATION_DIR <- here(snakemake@params[["saturation_dir"]])
ORA_ROOT <- here(snakemake@params[["ora_root"]])
stopifnot(nrow(fread(snakemake@input[["saturation_long"]])) > 0)

FDR <- 0.05

summary_paths <- list.files(ORA_ROOT, pattern = "^summary\\.csv$", recursive = TRUE, full.names = TRUE)
canonical <- "/rs[0-9]+/k[0-9]+/seed[0-9]+/[^./]+/[^./]+/summary\\.csv$"
summary_paths <- grep(canonical, summary_paths, value = TRUE)
summary_paths <- summary_paths[!grepl("/reactome/", summary_paths, fixed = TRUE)]

saturation <- rbindlist(lapply(summary_paths, function(path) {
    s <- fread(path)
    parts <- strsplit(path, "/", fixed = TRUE)[[1]]
    k_part <- parts[length(parts) - 4L]
    stopifnot(grepl("^k[0-9]+$", k_part))
    s[, k := as.integer(sub("^k", "", k_part))]
    enrich <- fread(file.path(dirname(path), "enrichment.csv.gz"), select = c("ID", "p.adjust"))
    recovered <- if (nrow(enrich)) uniqueN(enrich[`p.adjust` < FDR, ID]) else 0L
    s[, `:=`(
        recovered_pathways = recovered,
        recovered_percent = 100 * recovered / eligible_pathways,
        fdr = FDR
    )]
    s
}), fill = TRUE)

stopifnot(saturation[, all(latent_variables == k)])

model_colors <- c(CLAMPfull = "#0072B2", CLAMPbase = "#777777")
saturation[, model := factor(model, levels = c("CLAMPfull", "CLAMPbase"))]
saturation[, .N, by = .(model, database)]

## Grid coverage

In [ ]:
saturation[, .(cells = uniqueN(paste(fraction, k, seed)), databases = uniqueN(database)),
           by = .(model)][order(model)]

## Recovery against K, one series per subsampling level

In [ ]:
sat_seed <- saturation[, .(
    recovered_pathways = sum(recovered_pathways),
    eligible_pathways = sum(eligible_pathways)
), by = .(fraction, k, seed, model)]
sat_seed[, k_label := factor(k, levels = sort(unique(k)))]
sat_seed[, fraction_label := factor(sprintf("%d%%", fraction),
                                     levels = sprintf("%d%%", sort(unique(fraction))))]

DODGE <- position_dodge(width = 0.85)
dodge_offset <- function(g) {
    n <- nlevels(g)
    (as.integer(g) - (n + 1) / 2) * (0.85 / n)
}

make_k_plot <- function(model_name) {
    d <- sat_seed[model == model_name]
    if (!nrow(d)) return(invisible(NULL))
    means <- d[, .(recovered_pathways = mean(recovered_pathways)),
               by = .(fraction_label, k_label)]
    means[, x := as.integer(k_label) + dodge_offset(fraction_label)]
    ggplot(d, aes(k_label, recovered_pathways, fill = fraction_label, colour = fraction_label)) +
        geom_boxplot(position = DODGE, width = 0.75, outlier.shape = NA,
                     alpha = 0.35, linewidth = 0.4) +
        geom_point(position = position_jitterdodge(jitter.width = 0.12, dodge.width = 0.85),
                   size = 1.1, alpha = 0.65, show.legend = FALSE) +
        geom_line(data = means, aes(x = x, group = fraction_label),
                  linetype = "dashed", linewidth = 0.6, show.legend = FALSE) +
        scale_fill_viridis_d(end = 0.9, direction = -1) +
        scale_colour_viridis_d(end = 0.9, direction = -1) +
        scale_y_continuous(expand = expansion(mult = c(0.04, 0.08))) +
        labs(x = "Latent variables (K)",
             y = "Recovered pathways (combined, 3 databases)",
             fill = "Studies used", colour = "Studies used",
             title = model_name) +
        theme_classic(base_size = 15) +
        theme(panel.grid.major.y = element_line(colour = "#E3E3E3", linewidth = 0.3),
              plot.title = element_text(face = "bold", size = 14),
              legend.position = "right")
}

options(repr.plot.width = 13, repr.plot.height = 7)
make_k_plot("CLAMPfull")
make_k_plot("CLAMPbase")

## The full compendium against each smaller one, at the largest K

In [ ]:
LAST_K <- max(sat_seed$k)
REF <- tail(levels(sat_seed$fraction_label), 1L)

stars_for <- function(p) {
    ifelse(is.na(p), "n/a",
    ifelse(p < 0.001, "***",
    ifelse(p < 0.01, "**",
    ifelse(p < 0.05, "*", "ns"))))
}

ref_tests <- function(d) {
    dd <- droplevels(d[k == LAST_K])
    if (!REF %in% levels(dd$fraction_label)) return(data.table())
    y <- dd[fraction_label == REF, recovered_pathways]
    others <- setdiff(levels(dd$fraction_label), REF)
    if (!length(others) || length(y) < 2L) return(data.table())
    out <- rbindlist(lapply(others, function(f) {
        x <- dd[fraction_label == f, recovered_pathways]
        if (length(x) < 2L) {
            return(data.table(fraction_label = f, n = length(x), n_ref = length(y),
                              delta = NA_real_, p = NA_real_))
        }
        data.table(fraction_label = f, n = length(x), n_ref = length(y),
                   delta = mean(y) - mean(x),
                   p = t.test(y, x, alternative = "greater", var.equal = TRUE)$p.value)
    }))
    out[, p_adj := p.adjust(p, method = "BH")]
    out[, stars := stars_for(p_adj)]
    out[]
}

ref_k_tests <- rbindlist(lapply(levels(sat_seed$model), function(m) {
    r <- ref_tests(sat_seed[model == m])
    if (nrow(r)) r[, model := m]
    r
}), fill = TRUE)

make_ref_plot <- function(model_name) {
    d <- droplevels(sat_seed[model == model_name & k == LAST_K])
    st <- ref_k_tests[model == model_name]
    if (!nrow(d) || !nrow(st)) {
        message(sprintf("%s: not enough cells at K=%d to compare against %s yet.",
                        model_name, LAST_K, REF))
        return(invisible(NULL))
    }
    lv <- levels(d$fraction_label)
    span <- range(d$recovered_pathways)
    yr <- diff(span)
    st <- copy(st)[!is.na(delta)][order(match(fraction_label, lv))]
    st[, `:=`(x = match(as.character(fraction_label), lv), xend = match(REF, lv))]
    st[, y := span[2] + yr * (0.04 + 0.052 * seq_len(.N))]
    fmt_p <- function(p) ifelse(p < 0.001, sprintf("%.1e", p), sprintf("%.3f", p))
    st[, label := sprintf("q = %s  (%+.0f)", fmt_p(p_adj), delta)]

    ggplot(d, aes(fraction_label, recovered_pathways, fill = fraction_label)) +
        geom_boxplot(width = 0.6, outlier.shape = NA, alpha = 0.35, linewidth = 0.4,
                     show.legend = FALSE) +
        geom_jitter(width = 0.12, size = 1.6, alpha = 0.75, show.legend = FALSE) +
        geom_segment(data = st, aes(x = x, xend = xend, y = y, yend = y),
                     inherit.aes = FALSE, colour = "#444444", linewidth = 0.4) +
        geom_segment(data = st, aes(x = x, xend = x, y = y, yend = y - yr * 0.02),
                     inherit.aes = FALSE, colour = "#444444", linewidth = 0.4) +
        geom_segment(data = st, aes(x = xend, xend = xend, y = y, yend = y - yr * 0.02),
                     inherit.aes = FALSE, colour = "#444444", linewidth = 0.4) +
        geom_text(data = st, aes(x = (x + xend) / 2, y = y + yr * 0.016, label = label),
                  inherit.aes = FALSE, size = 4, colour = "#222222") +
        scale_fill_viridis_d(end = 0.9, direction = -1) +
        scale_y_continuous(expand = expansion(mult = c(0.08, 0.06))) +
        labs(x = "Studies used", y = "Recovered pathways (combined, 3 databases)",
             title = model_name) +
        theme_classic(base_size = 15) +
        theme(panel.grid.major.y = element_line(colour = "#E3E3E3", linewidth = 0.3),
              plot.title = element_text(face = "bold", size = 14))
}

options(repr.plot.width = 10, repr.plot.height = 7.5)
make_ref_plot("CLAMPfull")
make_ref_plot("CLAMPbase")

In [ ]:
ref_k_tests[, .(model, comparison = sprintf("%s vs %s", REF, fraction_label),
                n_ref, n, delta = round(delta, 1), p = signif(p, 3),
                p_adj = signif(p_adj, 3), stars)][order(model)]
